# Notebook 2 — Fine-tuning de modelos YOLO

**TP Visión por Computadora II — CEIA FIUBA**

Se entrenan tres variantes de YOLO mediante fine-tuning sobre el dataset Construction-PPE.
Los tres parten de pesos preentrenados en COCO (transfer learning).


In [ ]:
import importlib
import os
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import torch
import shutil
import time
from pathlib import Path
from collections import Counter, defaultdict, deque
from ultralytics import YOLO

EN_COLAB = importlib.util.find_spec("google.colab") is not None

plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid", palette="husl")

carpeta_datos = Path("data") if EN_COLAB else Path("../data")
carpeta_modelos = Path("models") if EN_COLAB else Path("../models")
carpeta_runs = Path("runs") if EN_COLAB else Path("../runs")

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
tamanio_imagen = 640 if dispositivo == "cuda" else 416

with open(carpeta_datos / "dataset_config.json") as f:
    config_dataset = json.load(f)

ruta_yaml_dataset = config_dataset["dataset_yaml"]
nombres_clases = config_dataset["class_names"]
cantidad_clases = config_dataset["num_classes"]

print(f"Dispositivo: {dispositivo}")
print(f"Clases: {nombres_clases}")


---
## Parte 2 — Fine-tuning de modelos YOLO

Se entrenan tres variantes de YOLO mediante fine-tuning sobre el dataset Construction-PPE.
Los tres parten de pesos preentrenados en COCO (transfer learning).

### Configuración del entrenamiento

In [12]:
import shutil
import time
from ultralytics import YOLO

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
tamanio_imagen = 640 if dispositivo == "cuda" else 416
tamanio_batch = 16 if dispositivo == "cuda" else 4
cantidad_epocas = 50 if dispositivo == "cuda" else 15
num_workers = 4 if dispositivo == "cuda" else 0

carpeta_runs = Path("runs")
carpeta_modelos = Path("models")
carpeta_modelos.mkdir(exist_ok=True)

print(f"Dispositivo:        {dispositivo}")
print(f"Tamaño de imagen:   {tamanio_imagen}x{tamanio_imagen}")
print(f"Tamaño de batch:    {tamanio_batch}")
print(f"Épocas:             {cantidad_epocas}")

Dispositivo:        cuda
Tamaño de imagen:   640x640
Tamaño de batch:    16
Épocas:             50


### Entrenamiento de YOLOv8n (Nano)

YOLOv8n es la variante más liviana (~3M parámetros). Se usa `patience=10` para early stopping.

In [13]:
print("=" * 60)
print("  Entrenando YOLOv8n (nano)")
print("=" * 60)

modelo_v8n = YOLO("yolov8n.pt")

resultados_v8n = modelo_v8n.train(
    data=ruta_yaml_dataset,
    epochs=cantidad_epocas,
    imgsz=tamanio_imagen,
    batch=tamanio_batch,
    device=dispositivo,
    workers=num_workers,
    project=str(carpeta_runs),
    name="yolov8n_ppe",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

# Usar save_dir del resultado para obtener la ruta real donde YOLO guardó los pesos
dir_v8n = Path(resultados_v8n.save_dir)
ruta_mejor_v8n = dir_v8n / "weights" / "best.pt"
shutil.copy(ruta_mejor_v8n, carpeta_modelos / "yolov8n_best.pt")
print(f"\nYOLOv8n guardado en: {carpeta_modelos / 'yolov8n_best.pt'}")
print(f"Directorio de run: {dir_v8n}")

  Entrenando YOLOv8n (nano)
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/construction-ppe/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_ppe, nbs=64, nms=False, opset=None, optimize=F

### Entrenamiento de YOLOv8m (Medium)

YOLOv8m tiene mayor capacidad (~25M parámetros). Batch reducido a la mitad por el mayor tamaño.

In [14]:
print("=" * 60)
print("  Entrenando YOLOv8m (medium)")
print("=" * 60)

modelo_v8m = YOLO("yolov8m.pt")

resultados_v8m = modelo_v8m.train(
    data=ruta_yaml_dataset,
    epochs=cantidad_epocas,
    imgsz=tamanio_imagen,
    batch=max(2, tamanio_batch // 2),
    device=dispositivo,
    workers=num_workers,
    project=str(carpeta_runs),
    name="yolov8m_ppe",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

dir_v8m = Path(resultados_v8m.save_dir)
ruta_mejor_v8m = dir_v8m / "weights" / "best.pt"
shutil.copy(ruta_mejor_v8m, carpeta_modelos / "yolov8m_best.pt")
print(f"\nYOLOv8m guardado en: {carpeta_modelos / 'yolov8m_best.pt'}")
print(f"Directorio de run: {dir_v8m}")

  Entrenando YOLOv8m (medium)
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/construction-ppe/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_ppe, nbs=64, nms=False, opset=None, optimize=

### Entrenamiento de YOLOv11n (Nano, arquitectura v11)

Nueva arquitectura con mejoras en el cuello de botella y cabezal de detección.

In [15]:
print("=" * 60)
print("  Entrenando YOLOv11n (YOLO11 nano)")
print("=" * 60)

modelo_v11n = YOLO("yolo11n.pt")

resultados_v11n = modelo_v11n.train(
    data=ruta_yaml_dataset,
    epochs=cantidad_epocas,
    imgsz=tamanio_imagen,
    batch=tamanio_batch,
    device=dispositivo,
    workers=num_workers,
    project=str(carpeta_runs),
    name="yolov11n_ppe",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

dir_v11n = Path(resultados_v11n.save_dir)
ruta_mejor_v11n = dir_v11n / "weights" / "best.pt"
shutil.copy(ruta_mejor_v11n, carpeta_modelos / "yolov11n_best.pt")
print(f"\nYOLOv11n guardado en: {carpeta_modelos / 'yolov11n_best.pt'}")
print(f"Directorio de run: {dir_v11n}")

  Entrenando YOLOv11n (YOLO11 nano)
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/construction-ppe/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11n_ppe, nbs=64, nms=False, opset=None, o

### Evaluación en el set de validación

In [16]:
rutas_modelos = {
    "YOLOv8n": carpeta_modelos / "yolov8n_best.pt",
    "YOLOv8m": carpeta_modelos / "yolov8m_best.pt",
    "YOLOv11n": carpeta_modelos / "yolov11n_best.pt",
}

resultados_evaluacion = {}

for nombre_modelo, ruta_modelo in rutas_modelos.items():
    if not ruta_modelo.exists():
        print(f"  {nombre_modelo}: no encontrado, se omite.")
        continue

    print(f"\nEvaluando {nombre_modelo}...")
    modelo = YOLO(str(ruta_modelo))

    inicio = time.time()
    metricas = modelo.val(
        data=ruta_yaml_dataset,
        imgsz=tamanio_imagen,
        device=dispositivo,
        workers=num_workers,
        verbose=False,
    )
    tiempo_evaluacion = time.time() - inicio

    resultados_evaluacion[nombre_modelo] = {
        "map50": round(float(metricas.box.map50), 4),
        "map50_95": round(float(metricas.box.map), 4),
        "precision": round(float(metricas.box.mp), 4),
        "recall": round(float(metricas.box.mr), 4),
        "val_time_s": round(tiempo_evaluacion, 1),
    }

    r = resultados_evaluacion[nombre_modelo]
    print(f"  mAP50:      {r['map50']:.4f}")
    print(f"  mAP50-95:   {r['map50_95']:.4f}")
    print(f"  Precision:  {r['precision']:.4f}")
    print(f"  Recall:     {r['recall']:.4f}")

with open(carpeta_datos / "eval_results.json", "w") as f:
    json.dump(resultados_evaluacion, f, indent=2)

print("\nResultados guardados en data/eval_results.json")


Evaluando YOLOv8n...
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 73 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1957.4±435.5 MB/s, size: 59.5 KB)
val: Scanning /content/data/construction-ppe/valid/labels.cache... 114 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 114/114 47.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.4it/s 2.3s
                   all        114        697      0.896      0.685      0.777      0.476
Speed: 2.2ms preprocess, 1.9ms inference, 0.0ms loss, 6.3ms postprocess per image
Results saved to /content/runs/detect/val
  mAP50:      0.7773
  mAP50-95:   0.4764
  Precision:  0.8964
  Recall:     0.6851

Evaluando YOLOv8m...
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 93 layer

### Benchmark de latencia de inferencia

In [17]:
imagen_sintetica = np.random.randint(0, 255, (tamanio_imagen, tamanio_imagen, 3), dtype=np.uint8)
cantidad_mediciones = 20

resultados_latencia = {}

for nombre_modelo, ruta_modelo in rutas_modelos.items():
    if not ruta_modelo.exists():
        continue

    modelo = YOLO(str(ruta_modelo))

    for _ in range(3):
        _ = modelo.predict(imagen_sintetica, verbose=False, device=dispositivo)

    tiempos_ms = []
    for _ in range(cantidad_mediciones):
        t0 = time.perf_counter()
        _ = modelo.predict(imagen_sintetica, verbose=False, device=dispositivo)
        tiempos_ms.append((time.perf_counter() - t0) * 1000)

    resultados_latencia[nombre_modelo] = {
        "mean_ms": round(np.mean(tiempos_ms), 1),
        "std_ms": round(np.std(tiempos_ms), 1),
        "p50_ms": round(np.percentile(tiempos_ms, 50), 1),
        "p95_ms": round(np.percentile(tiempos_ms, 95), 1),
        "fps": round(1000 / np.mean(tiempos_ms), 1),
    }

    r = resultados_latencia[nombre_modelo]
    print(f"{nombre_modelo}: {r['mean_ms']:.1f} ms +/- {r['std_ms']:.1f} | {r['fps']:.1f} FPS")

with open(carpeta_datos / "latency_results.json", "w") as f:
    json.dump(resultados_latencia, f, indent=2)

print("\nLatencias guardadas en data/latency_results.json")

YOLOv8n: 8.3 ms +/- 0.1 | 119.9 FPS
YOLOv8m: 10.3 ms +/- 0.1 | 97.4 FPS
YOLOv11n: 10.3 ms +/- 0.2 | 97.2 FPS

Latencias guardadas en data/latency_results.json


In [18]:
if EN_COLAB:
    from google.colab import files

    archivos_a_descargar = [
        carpeta_modelos / "yolov8n_best.pt",
        carpeta_modelos / "yolov8m_best.pt",
        carpeta_modelos / "yolov11n_best.pt",
        carpeta_datos / "eval_results.json",
        carpeta_datos / "latency_results.json",
        carpeta_datos / "dataset_config.json",
    ]

    for archivo in archivos_a_descargar:
        if archivo.exists():
            print(f"Descargando {archivo.name}...")
            files.download(str(archivo))
        else:
            print(f"  {archivo.name} no encontrado, se omite.")
else:
    print("Entorno local — los archivos ya estan en disco.")
